In [7]:
def load_chunks_from_csv(csv_path: Path) -> Tuple[List[Dict], pd.DataFrame]:
    """
    Load chunk records from the cleaned_formatted_chunks.csv generated by data pipeline.
    Returns list of chunk dicts and DataFrame for reference.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"Chunk file not found: {csv_path}")
    
    df = pd.read_csv(csv_path)
    chunks = df.to_dict('records')
    
    # Parse accessible_roles from comma-separated string to list
    for chunk in chunks:
        if isinstance(chunk['accessible_roles'], str):
            chunk['accessible_roles'] = [r.strip() for r in chunk['accessible_roles'].split(',')]
    
    return chunks, df

# Load chunks
csv_path = CHUNKS_DIR / "cleaned_formatted_chunks.csv"
chunks, chunks_df = load_chunks_from_csv(csv_path)

print(f"✅ Loaded {len(chunks)} chunks from CSV")
print(f"\n📋 Chunk schema:")
print(f"   Columns: {list(chunks_df.columns)}")
print(f"   Sample chunk:")
for key, val in list(chunks[0].items())[:6]:
    preview = str(val)[:60] + "..." if len(str(val)) > 60 else str(val)
    print(f"     {key}: {preview}")

✅ Loaded 311 chunks from CSV

📋 Chunk schema:
   Columns: ['chunk_id', 'source_document', 'source_path', 'department', 'doc_type', 'chunk_sequence', 'token_count', 'accessible_roles', 'content']
   Sample chunk:
     chunk_id: CHUNK-000001
     source_document: engineering_master_doc.md
     source_path: D:\Projects\RBAC\data\engineering\engineering_master_doc.md
     department: engineering
     doc_type: md
     chunk_sequence: 1


In [8]:
print("🤖 Initializing embedding model...")
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

if EMBEDDING_BACKEND == "fastembed":
    print(f"   Model: {embedding_model_name}")
    print("   Backend: fastembed (ONNX)")
    model = TextEmbedding(model_name=embedding_model_name)

    test_text = "This is a test embedding"
    test_embedding = np.array(list(model.embed([test_text]))[0])
    embedding_dim = int(test_embedding.shape[0])

else:
    print(f"   Model request: {embedding_model_name}")
    print("   Backend: TF-IDF fallback (runtime DLL dependency missing for ONNX/Torch)")
    model = TfidfVectorizer(max_features=384, ngram_range=(1, 2))
    test_embedding = np.zeros(384, dtype=np.float32)
    embedding_dim = 384

print("\n✅ Embedding backend initialized!")
print(f"   Effective dimensions: {embedding_dim}")
print(f"   Test embedding shape: {test_embedding.shape}")
print(f"   First 5 dimensions: {test_embedding[:5]}")

🤖 Initializing embedding model...
   Model request: sentence-transformers/all-MiniLM-L6-v2
   Backend: TF-IDF fallback (runtime DLL dependency missing for ONNX/Torch)

✅ Embedding backend initialized!
   Effective dimensions: 384
   Test embedding shape: (384,)
   First 5 dimensions: [0. 0. 0. 0. 0.]


In [11]:
print("🔄 Generating embeddings for all chunks...")
print(f"   Processing {len(chunks)} chunks...\n")

embedding_start_time = time.time()

# Extract chunk content for embedding
chunk_contents = [chunk['content'] for chunk in chunks]

# Generate embeddings using selected backend
if EMBEDDING_BACKEND == "fastembed":
    embeddings = np.array(list(model.embed(chunk_contents)), dtype=np.float32)
else:
    # TF-IDF fallback path
    embeddings_sparse = model.fit_transform(chunk_contents)
    embeddings = embeddings_sparse.toarray().astype(np.float32)

embedding_elapsed_time = time.time() - embedding_start_time

# Add embeddings to chunk records
for i, chunk in enumerate(chunks):
    chunk['embedding'] = embeddings[i]

print(f"\n✅ Embeddings generated successfully!")
print(f"   Backend: {EMBEDDING_BACKEND}")
print(f"   Total chunks: {len(chunks)}")
print(f"   Embedding shape: {embeddings.shape}")
print(f"   Time elapsed: {embedding_elapsed_time:.2f}s")
print(f"   Average time/chunk: {(embedding_elapsed_time/len(chunks))*1000:.2f}ms")

# Calculate statistics
embedding_norms = np.linalg.norm(embeddings, axis=1)
print(f"\n📊 Embedding statistics:")
print(f"   Min norm: {embedding_norms.min():.4f}")
print(f"   Max norm: {embedding_norms.max():.4f}")
print(f"   Mean norm: {embedding_norms.mean():.4f}")

🔄 Generating embeddings for all chunks...
   Processing 311 chunks...


✅ Embeddings generated successfully!
   Backend: tfidf
   Total chunks: 311
   Embedding shape: (311, 384)
   Time elapsed: 0.04s
   Average time/chunk: 0.12ms

📊 Embedding statistics:
   Min norm: 1.0000
   Max norm: 1.0000
   Mean norm: 1.0000


In [12]:
print("🗄️  Initializing Chroma vector database...")

# Initialize persistent Chroma client
chroma_client = chromadb.PersistentClient(path=str(DB_DIR))

# Remove existing collection if it exists (for clean rebuild)
try:
    chroma_client.delete_collection("documents")
except Exception:
    pass

# Create new collection with cosine similarity
collection = chroma_client.get_or_create_collection(
    name="documents",
    metadata={"hnsw:space": "cosine"},
)

print("✅ Chroma initialized!")
print(f"   Storage: {DB_DIR}")
print(f"   Collection: 'documents'")
print("   Distance metric: cosine")

# Index chunks with embeddings and metadata
print("\n📇 Indexing chunks into vector database...")
print(f"   Total chunks: {len(chunks)}")

# Prepare data for indexing
chunk_ids = [chunk['chunk_id'] for chunk in chunks]
embeddings_list = [chunk['embedding'].tolist() for chunk in chunks]
metadatas = []

for chunk in chunks:
    metadata = {
        'source_document': chunk['source_document'],
        'source_path': chunk['source_path'],
        'department': chunk['department'],
        'doc_type': chunk['doc_type'],
        'chunk_sequence': str(chunk['chunk_sequence']),
        'token_count': str(chunk['token_count']),
        'accessible_roles': '|'.join(chunk['accessible_roles']),
    }
    metadatas.append(metadata)

# Add chunks to collection
index_start_time = time.time()
collection.add(
    ids=chunk_ids,
    embeddings=embeddings_list,
    metadatas=metadatas,
    documents=[chunk['content'] for chunk in chunks],
)
index_elapsed_time = time.time() - index_start_time

print("\n✅ Indexing complete!")
print(f"   Chunks indexed: {len(chunk_ids)}")
print(f"   Time elapsed: {index_elapsed_time:.2f}s")

# Verify indexing
indexed_count = collection.count()
print(f"   Verified in DB: {indexed_count} chunks")

🗄️  Initializing Chroma vector database...
✅ Chroma initialized!
   Storage: d:\Projects\RBAC\vector_db
   Collection: 'documents'
   Distance metric: cosine

📇 Indexing chunks into vector database...
   Total chunks: 311

✅ Indexing complete!
   Chunks indexed: 311
   Time elapsed: 0.18s
   Verified in DB: 311 chunks


In [17]:
def _embed_query(query: str, model, backend: str) -> List[float]:
    if backend == "fastembed":
        return np.array(list(model.embed([query]))[0], dtype=np.float32).tolist()

    # TF-IDF fallback query encoding
    query_vec = model.transform([query]).toarray()[0].astype(np.float32)
    return query_vec.tolist()


def semantic_search(
    query: str,
    collection,
    model,
    top_k: int = 5,
    user_roles: List[str] = None,
    department_filter: str = None,
) -> List[Dict]:
    """Perform semantic search with optional RBAC and department filtering."""

    query_embedding = _embed_query(query, model, EMBEDDING_BACKEND)

    where_filter = None
    if department_filter:
        where_filter = {"department": {"$eq": department_filter}}

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k * 3,
        where=where_filter,
        include=["documents", "metadatas", "distances"],
    )

    search_results = []
    for i in range(len(results["ids"][0])):
        metadata = results["metadatas"][0][i]
        accessible_roles = metadata["accessible_roles"].split("|")

        if user_roles and not any(role in accessible_roles for role in user_roles):
            continue

        distance = float(results["distances"][0][i])
        similarity = max(0.0, min(1.0, 1.0 - distance))

        search_results.append(
            {
                "chunk_id": results["ids"][0][i],
                "similarity": round(similarity, 4),
                "content": results["documents"][0][i],
                "source_document": metadata["source_document"],
                "source_path": metadata["source_path"],
                "department": metadata["department"],
                "doc_type": metadata["doc_type"],
                "chunk_sequence": int(metadata["chunk_sequence"]),
                "token_count": int(metadata["token_count"]),
                "accessible_roles": accessible_roles,
            }
        )

    return search_results[:top_k]

print("✅ Semantic search function defined!")
print(f"   Active backend: {EMBEDDING_BACKEND}")
print("\nFunction signature:")
print("  semantic_search(query, collection, model, top_k=5, user_roles=None, department_filter=None)")

✅ Semantic search function defined!
   Active backend: tfidf

Function signature:
  semantic_search(query, collection, model, top_k=5, user_roles=None, department_filter=None)


In [18]:
print("🔍 Testing semantic search with example queries...\n")

# Test 1: Basic semantic search
print("=" * 70)
print("TEST 1: Basic Semantic Search")
print("=" * 70)
query1 = "financial performance and revenue"
results1 = semantic_search(query1, collection, model, top_k=3)

print(f"Query: '{query1}'")
print(f"Results: {len(results1)} matches\n")
for i, result in enumerate(results1, 1):
    print(f"{i}. Chunk {result['chunk_id']} (Similarity: {result['similarity']})")
    print(f"   Document: {result['source_document']}")
    print(f"   Department: {result['department']}")
    print(f"   Preview: {result['content'][:80]}...\n")

# Test 2: RBAC-filtered search
print("=" * 70)
print("TEST 2: Search with Role-Based Access Control")
print("=" * 70)
query2 = "engineering best practices"
user_roles_eng = ["engineering", "tech_lead"]
results2 = semantic_search(query2, collection, model, top_k=3, user_roles=user_roles_eng)

print(f"Query: '{query2}'")
print(f"User roles: {user_roles_eng}")
print(f"Results: {len(results2)} matches\n")
for i, result in enumerate(results2, 1):
    print(f"{i}. Chunk {result['chunk_id']} (Similarity: {result['similarity']})")
    print(f"   Document: {result['source_document']}")
    print(f"   Accessible roles: {result['accessible_roles']}")
    print(f"   Preview: {result['content'][:80]}...\n")

# Test 3: Department-filtered search
print("=" * 70)
print("TEST 3: Department-Filtered Search")
print("=" * 70)
query3 = "marketing strategy"
dept_filter = "marketing"
results3 = semantic_search(query3, collection, model, top_k=3, department_filter=dept_filter)

print(f"Query: '{query3}'")
print(f"Department filter: {dept_filter}")
print(f"Results: {len(results3)} matches\n")
for i, result in enumerate(results3, 1):
    print(f"{i}. Chunk {result['chunk_id']} (Similarity: {result['similarity']})")
    print(f"   Document: {result['source_document']}")
    print(f"   Preview: {result['content'][:80]}...\n")

print("✅ Search testing complete!")

🔍 Testing semantic search with example queries...

TEST 1: Basic Semantic Search
Query: 'financial performance and revenue'
Results: 3 matches

1. Chunk CHUNK-000107 (Similarity: 0.407)
   Document: quarterly_financial_report.md
   Department: finance
   Preview: accounts payable processing. - mitigation: streamlined payment workflows and ren...

2. Chunk CHUNK-000119 (Similarity: 0.3537)
   Document: quarterly_financial_report.md
   Department: finance
   Preview: accelerate receivables. ---  q4 - october to december 2024  quarterly financial ...

3. Chunk CHUNK-000101 (Similarity: 0.321)
   Document: quarterly_financial_report.md
   Department: finance
   Preview: ment to sustainable growth and shareholder value. ---  q1 - january to march 202...

TEST 2: Search with Role-Based Access Control
Query: 'engineering best practices'
User roles: ['engineering', 'tech_lead']
Results: 3 matches

1. Chunk CHUNK-000002 (Similarity: 0.4571)
   Document: engineering_master_doc.md
   Accessible r

In [19]:
print("📊 Performance and Quality Benchmarking...\n")

# Benchmark queries covering different departments and topics
benchmark_queries = [
    ("financial quarterly results", None, None),
    ("engineering architecture", ["engineering", "tech_lead"], None),
    ("employee handbook policies", ["hr"], "general"),
    ("marketing strategy and campaigns", None, "marketing"),
    ("quarterly performance review", ["finance", "leadership"], None),
    ("technical documentation", ["engineering"], None),
    ("company benefits", None, "hr"),
    ("market analysis and trends", ["marketing", "sales"], "marketing"),
    ("product roadmap", ["engineering", "product"], None),
    ("financial forecasting", ["finance", "leadership"], None),
]

benchmark_results = []
print("Running benchmark queries...\n")

for i, (query, roles, dept) in enumerate(benchmark_queries, 1):
    start_time = time.time()
    results = semantic_search(query, collection, model, top_k=5, user_roles=roles, department_filter=dept)
    elapsed_time = (time.time() - start_time) * 1000

    benchmark_results.append({
        'query': query,
        'user_roles': str(roles),
        'department_filter': dept,
        'results_found': len(results),
        'query_time_ms': round(elapsed_time, 2),
        'top_similarity': results[0]['similarity'] if results else 0,
    })

    print(f"{i}. '{query}'")
    if roles:
        print(f"   Roles: {roles}")
    if dept:
        print(f"   Dept: {dept}")
    print(f"   ✓ {len(results)} results | {elapsed_time:.2f}ms | Top sim: {benchmark_results[-1]['top_similarity']}")

benchmark_df = pd.DataFrame(benchmark_results)

print("\n" + "=" * 70)
print("BENCHMARK RESULTS SUMMARY")
print("=" * 70)
print(benchmark_df.to_string(index=False))

print("\n📈 Performance Metrics:")
print(f"   Average query time: {benchmark_df['query_time_ms'].mean():.2f}ms")
print(f"   Min query time: {benchmark_df['query_time_ms'].min():.2f}ms")
print(f"   Max query time: {benchmark_df['query_time_ms'].max():.2f}ms")
print(f"   Median query time: {benchmark_df['query_time_ms'].median():.2f}ms")

print(f"\n🎯 Search Quality Metrics:")
print(f"   Queries returning results: {(benchmark_df['results_found'] > 0).sum()}/{len(benchmark_df)}")
print(f"   Average results per query: {benchmark_df['results_found'].mean():.1f}")
print(f"   Average top similarity: {benchmark_df['top_similarity'].mean():.4f}")

print("\n💾 Database persisted at:")
print(f"   {DB_DIR}")

📊 Performance and Quality Benchmarking...

Running benchmark queries...

1. 'financial quarterly results'
   ✓ 5 results | 3.36ms | Top sim: 0.3163
2. 'engineering architecture'
   Roles: ['engineering', 'tech_lead']
   ✓ 5 results | 0.00ms | Top sim: 0.5416
3. 'employee handbook policies'
   Roles: ['hr']
   Dept: general
   ✓ 0 results | 12.12ms | Top sim: 0
4. 'marketing strategy and campaigns'
   Dept: marketing
   ✓ 5 results | 4.22ms | Top sim: 0.3026
5. 'quarterly performance review'
   Roles: ['finance', 'leadership']
   ✓ 5 results | 0.00ms | Top sim: 0.2151
6. 'technical documentation'
   Roles: ['engineering']
   ✓ 5 results | 6.47ms | Top sim: 0.528
7. 'company benefits'
   Dept: hr
   ✓ 5 results | 2.03ms | Top sim: 0.0
8. 'market analysis and trends'
   Roles: ['marketing', 'sales']
   Dept: marketing
   ✓ 5 results | 4.17ms | Top sim: 0.8061
9. 'product roadmap'
   Roles: ['engineering', 'product']
   ✓ 4 results | 0.00ms | Top sim: 0.4335
10. 'financial forecasting'
   

In [20]:
print("📄 Generating comprehensive benchmark report...\n")

# Create detailed report
report_md = f"""# Vector Database & Semantic Search Benchmarking Report

**Generated:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

## 1. System Configuration

### Embedding Model
- **Requested model:** sentence-transformers/all-MiniLM-L6-v2
- **Active backend:** {EMBEDDING_BACKEND}
- **Dimensions:** {embeddings.shape[1]}

### Vector Database
- **System:** Chroma
- **Storage:** Persistent local storage
- **Distance metric:** Cosine similarity

### Data Statistics
- **Total chunks indexed:** {collection.count()}
- **Total content volume:** {sum([len(chunk['content']) for chunk in chunks]):,} characters
- **Average chunk size:** {np.mean([len(chunk['content']) for chunk in chunks]):.0f} characters
- **Departments:** {len(chunks_df['department'].unique())}
- **Source documents:** {len(chunks_df['source_document'].unique())}

## 2. Embedding Generation Performance

- **Total time:** {embedding_elapsed_time:.2f} seconds
- **Average time per chunk:** {(embedding_elapsed_time/len(chunks))*1000:.2f}ms
- **Throughput:** {len(chunks)/embedding_elapsed_time:.1f} chunks/second

## 3. Indexing Performance

- **Chunks indexed:** {collection.count()}
- **Total indexing time:** {index_elapsed_time:.2f} seconds

## 4. Query Performance Benchmarks

### Benchmark Query Results

```
{benchmark_df.to_string(index=False)}
```

### Performance Summary

| Metric | Value |
|--------|-------|
| Average Query Time | {benchmark_df['query_time_ms'].mean():.2f}ms |
| Min Query Time | {benchmark_df['query_time_ms'].min():.2f}ms |
| Max Query Time | {benchmark_df['query_time_ms'].max():.2f}ms |
| Median Query Time | {benchmark_df['query_time_ms'].median():.2f}ms |

### Search Quality Metrics

| Metric | Value |
|--------|-------|
| Queries with Results | {(benchmark_df['results_found'] > 0).sum()}/{len(benchmark_df)} |
| Average Results per Query | {benchmark_df['results_found'].mean():.1f} |
| Average Top Similarity | {benchmark_df['top_similarity'].mean():.4f} |

## 5. Notes

- If backend is `tfidf`, install Microsoft Visual C++ Redistributable to enable ONNX/Torch runtimes on Windows.
- Once runtime dependencies are fixed, rerun the notebook to use all-MiniLM embeddings.
"""

# Save report
report_path = ARTIFACTS_DIR / "embedding_benchmark_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_md)

print(f"✅ Report saved to: {report_path}\n")

# Also save benchmark data as CSV
benchmark_path = ARTIFACTS_DIR / "benchmark_results.csv"
benchmark_df.to_csv(benchmark_path, index=False)
print(f"✅ Benchmark data saved to: {benchmark_path}\n")

print("=" * 70)
print("PIPELINE COMPLETION SUMMARY")
print("=" * 70)
print(f"📊 Chunks processed: {len(chunks)}")
print(f"🔢 Embedding dimension: {embeddings.shape[1]}")
print(f"🗄️  Vector DB status: READY ({collection.count()} indexed)")
print(f"🔍 Semantic search: ENABLED")
print(f"⚡ Query performance: {benchmark_df['query_time_ms'].mean():.1f}ms avg")
print(f"📄 Benchmark report: Generated")
print(f"💾 Database storage: {DB_DIR}")
print("=" * 70)

📄 Generating comprehensive benchmark report...

✅ Report saved to: d:\Projects\RBAC\artifacts\module2\embedding_benchmark_report.md

✅ Benchmark data saved to: d:\Projects\RBAC\artifacts\module2\benchmark_results.csv

PIPELINE COMPLETION SUMMARY
📊 Chunks processed: 311
🔢 Embedding dimension: 384
🗄️  Vector DB status: READY (311 indexed)
🔍 Semantic search: ENABLED
⚡ Query performance: 3.8ms avg
📄 Benchmark report: Generated
💾 Database storage: d:\Projects\RBAC\vector_db
